# Validate the differentiable sky API

This notebook uses tiny deterministic arrays so it runs quickly on CPU. It checks the public output shape, JIT compilation, and a gradient through the complete radiative-transfer calculation.

In [ ]:
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
from lowsky import SkyInputs, SkyParameters, generate_sky

In [ ]:
pixels, distances = 12, 6
inputs = SkyInputs(
    emissivity_408=jnp.linspace(1.0, 2.0, pixels * distances).reshape(pixels, distances),
    emission_measure_rate=jnp.full((pixels, distances), 0.35),
    synchrotron_multiplier=jnp.ones((pixels, distances)),
    spectral_index=jnp.linspace(-2.4, -2.7, pixels),
    shell_emission_408=jnp.full((2, 2, pixels), 0.08),
    shell_foreground_emission_measure=jnp.full((2, 2, pixels), 0.2),
    shell_spectral_index=jnp.array([-2.7, -2.9]),
    shell_low_frequency_spectral_index=jnp.array([-2.5, -2.6]),
    distance_step_kpc=jnp.asarray(0.25),
)
frequencies = jnp.array([3.0, 10.0, 30.0, 50.0])
parameters = SkyParameters()

In [ ]:
compiled_generate = jax.jit(generate_sky)
sky = compiled_generate(frequencies, inputs, parameters)
gradient = jax.grad(
    lambda scale: jnp.mean(generate_sky(
        frequencies, inputs, parameters._replace(emissivity_scale=scale)
    ))
)(jnp.asarray(1.0))
assert sky.shape == (len(frequencies), pixels)
assert jnp.all(jnp.isfinite(sky)) and jnp.isfinite(gradient)
print(f"mean-temperature gradient = {float(gradient):.3e} K")

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 3.5))
ax.loglog(frequencies, jnp.mean(sky, axis=1), marker="o")
ax.set(xlabel="Frequency [MHz]", ylabel="Mean temperature [K]", title="Toy differentiable sky")
ax.grid(True, which="both", alpha=0.25);